<a href="https://colab.research.google.com/github/yenlung/Python-Math-AI/blob/main/12%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%85%A5%E9%96%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12 機器學習入門 🤖
## 分類 × 聚類 × AI 的兩大學習方式

上週我們學了線性迴歸（預測數字），這週我們要解決更有趣的問題：

### 機器學習的兩大類型：

| 類型 | 說明 | 例子 |
|------|------|------|
| **監督式學習** (Supervised) | 有正確答案的訓練數據 | 分辨貓狗、垃圾郵件偵測 |
| **非監督式學習** (Unsupervised) | 沒有正確答案，自己找規律 | 客群分析、新聞分類 |

---

### 今天的主角：

1. **SVM 分類器** — 監督式學習的經典算法
2. **K-Means 聚類** — 非監督式學習的入門算法
3. **DBSCAN** — 更聰明的聚類算法，可以找出「異形」群

## 🤖 AI 可以怎麼幫你？

你可以問 AI：

- SVM 的原理是什麼？
- 監督式和非監督式學習有什麼差別？
- K-Means 怎麼決定分幾群？
- 分類正確率 70% 算高還是低？
- DBSCAN 的 `eps` 和 `min_samples` 怎麼設？

但記得：**先自己想，再問 AI**

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

## 1. 監督式學習：鳶尾花分類

### 認識鳶尾花資料集

這是機器學習的「Hello World」！

1936 年由英國統計學家 Ronald Fisher 蒐集，包含三種鳶尾花：

- 🌸 **Setosa**
- 🌺 **Versicolor**
- 🌼 **Virginica**

每朵花有 4 個特徵：花萼長寬、花瓣長寬。

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()

# 轉成 DataFrame，方便探索
feature_names_zh = ['花萼長度', '花萼寬度', '花瓣長度', '花瓣寬度']
df = pd.DataFrame(data=iris.data, columns=feature_names_zh)
df['標籤'] = iris.target
df['花種名稱'] = df['標籤'].map({0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'})

print('資料大小:', df.shape)
df.head()

In [ ]:
# 每種花各有幾筆？
print(df['花種名稱'].value_counts())

### 視覺化：三種花能用肉眼分開嗎？

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
species = ['Setosa', 'Versicolor', 'Virginica']

for i, (sp, color) in enumerate(zip(species, colors)):
    mask = df['標籤'] == i
    axes[0].scatter(df[mask]['花萼長度'], df[mask]['花萼寬度'],
                    label=sp, color=color, alpha=0.7)
    axes[1].scatter(df[mask]['花瓣長度'], df[mask]['花瓣寬度'],
                    label=sp, color=color, alpha=0.7)

axes[0].set_xlabel('花萼長度')
axes[0].set_ylabel('花萼寬度')
axes[0].set_title('花萼特徵')
axes[0].legend()

axes[1].set_xlabel('花瓣長度')
axes[1].set_ylabel('花瓣寬度')
axes[1].set_title('花瓣特徵（更容易分！）')
axes[1].legend()

plt.suptitle('三種鳶尾花的特徵分佈', fontsize=14)
plt.tight_layout()
plt.show()

### SVM 分類器

**Support Vector Machine（支持向量機）** 是一個非常聰明的分類算法。

它的核心想法：在不同類別之間，找到一條**最大間隔**的分界線（超平面）。

我們先用「花萼長度」和「花萼寬度」兩個特徵來訓練，看看效果如何。

In [ ]:
from sklearn.model_selection import train_test_split

# 只用花萼長寬兩個特徵（方便視覺化）
X = df[['花萼長度', '花萼寬度']].values
Y = df['標籤'].values

x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=9487
)

print('訓練集:', x_train.shape)
print('測試集:', x_test.shape)

In [ ]:
# 先看看訓練資料的分佈
plt.figure(figsize=(7, 5))
scatter = plt.scatter(x_train[:, 0], x_train[:, 1],
                      c=y_train, cmap='coolwarm', alpha=0.8, edgecolors='white', s=60)
plt.colorbar(scatter, ticks=[0, 1, 2], label='花種')
plt.xlabel('花萼長度')
plt.ylabel('花萼寬度')
plt.title('訓練數據分佈（用顏色表示花種）')
plt.show()

In [ ]:
from sklearn.svm import SVC

# Step 1
clf = SVC(kernel='linear')

# Step 2
clf.fit(x_train, y_train)

# Step 3
y_pred = clf.predict(x_test)

print('訓練完成！')

### 評估：準確率

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

test_acc = accuracy_score(y_test, y_pred)
train_acc = accuracy_score(y_train, clf.predict(x_train))

print(f'訓練集準確率: {train_acc:.1%}')
print(f'測試集準確率: {test_acc:.1%}')

### 混淆矩陣：哪種花最容易搞混？

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=species, yticklabels=species)
plt.title('混淆矩陣')
plt.xlabel('預測類別')
plt.ylabel('真實類別')
plt.tight_layout()
plt.show()

print('對角線是預測正確的，其他是預測錯誤的。')

### 哪些花被分錯了？

In [ ]:
errors = y_pred - y_test  # 0 代表正確，非 0 代表錯誤

plt.figure(figsize=(7, 5))
plt.scatter(x_test[:, 0], x_test[:, 1],
            c=errors, cmap='RdYlGn', edgecolors='black',
            s=100, vmin=-1, vmax=1)
plt.xlabel('花萼長度')
plt.ylabel('花萼寬度')
plt.title('測試集預測結果（綠=正確，紅=錯誤）')
plt.show()

print(f'共 {len(y_test)} 筆測試，{int(test_acc * len(y_test))} 筆正確，{int((1-test_acc)*len(y_test))} 筆錯誤')

### 🤔 能不能做得更好？

剛才只用了 2 個特徵（花萼），現在加上花瓣試試！

In [ ]:
# 用全部 4 個特徵
X_all = df[['花萼長度', '花萼寬度', '花瓣長度', '花瓣寬度']].values
Y_all = df['標籤'].values

x_train2, x_test2, y_train2, y_test2 = train_test_split(
    X_all, Y_all, test_size=0.2, random_state=9487
)

clf2 = SVC(kernel='linear')
clf2.fit(x_train2, y_train2)
y_pred2 = clf2.predict(x_test2)

acc2 = accuracy_score(y_test2, y_pred2)
print(f'2 個特徵的測試準確率: {test_acc:.1%}')
print(f'4 個特徵的測試準確率: {acc2:.1%}')
print(f'特徵越多，效果越好！')

## 2. 非監督式學習：K-Means 聚類

### 什麼是聚類？

監督式學習：老師告訴你「這是貓、這是狗」
非監督式學習：沒有老師，自己把相似的東西歸在一起

**應用場景**：客群分析、新聞分類、基因分析、圖片壓縮...

In [ ]:
from sklearn.datasets import make_blobs

# 產生 3 群隨機數據（真實情況 AI 不知道有幾群）
x_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1, random_state=42)

# 先假裝我們不知道顏色（y_blobs），用灰色畫
plt.figure(figsize=(7, 5))
plt.scatter(x_blobs[:, 0], x_blobs[:, 1], c='gray', edgecolor='k', s=50, alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('未標記的數據——你能看出幾群嗎？')
plt.show()

### K-Means 算法的核心思想

1. 隨機放置 K 個「中心點」
2. 每個數據點找最近的中心點，分配到那一群
3. 更新中心點位置（群內平均）
4. 重複 2-3 直到收斂

**關鍵問題：K 要選幾？**（下面會教你！）

In [ ]:
from sklearn.cluster import KMeans

# 試試分 3 群
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(x_blobs)

plt.figure(figsize=(7, 5))
plt.scatter(x_blobs[:, 0], x_blobs[:, 1],
            c=kmeans.labels_, cmap='viridis', s=50, alpha=0.7, edgecolors='white')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c='red', marker='★', s=300, zorder=5, label='群中心')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('K-Means 分群結果（K=3）')
plt.legend()
plt.show()

### 手肘法：決定 K 值

怎麼知道該分幾群？用「手肘法」！

計算不同 K 值下的「群內距離平方和」(inertia)，
找到曲線開始「彎折」的地方（像手肘），那就是最佳的 K。

In [ ]:
inertias = []
K_range = range(1, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(x_blobs)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='最佳 K=3')
plt.xlabel('K（群數）')
plt.ylabel('群內距離平方和')
plt.title('手肘法：選擇最佳 K 值')
plt.legend()
plt.xticks(K_range)
plt.show()

print('👉 曲線在 K=3 後開始趨於平緩，K=3 是最佳選擇！')

### 不同 K 值的效果比較

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, k in enumerate([2, 3, 5, 8]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(x_blobs)

    axes[idx].scatter(x_blobs[:, 0], x_blobs[:, 1],
                      c=km.labels_, cmap='viridis', s=20, alpha=0.7)
    axes[idx].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                      c='red', s=200, marker='*')
    axes[idx].set_title(f'K = {k}')
    axes[idx].set_xticks([])
    axes[idx].set_yticks([])

plt.suptitle('不同 K 值的 K-Means 結果', fontsize=13)
plt.tight_layout()
plt.show()

## 3. DBSCAN：更聰明的聚類

K-Means 的限制：
- 需要**事先指定 K**
- 只適合**圓形**的群
- 對**離群值**很敏感

**DBSCAN（密度聚類）** 的優點：
- 自動決定群數
- 可以找到**任意形狀**的群
- 自動偵測**離群值**（標記為 -1）

In [ ]:
from sklearn.cluster import DBSCAN

# 先在剛才的 blob 數據上試試
dbscan = DBSCAN(eps=0.5, min_samples=3)
dbscan.fit(x_blobs)

n_clusters = len(set(dbscan.labels_)) - (1 if -1 in dbscan.labels_ else 0)
n_noise = list(dbscan.labels_).count(-1)

print(f'自動找到 {n_clusters} 個群')
print(f'偵測到 {n_noise} 個離群點')

plt.figure(figsize=(7, 5))
plt.scatter(x_blobs[:, 0], x_blobs[:, 1],
            c=dbscan.labels_, cmap='viridis', s=50, alpha=0.7, edgecolors='white')
plt.title(f'DBSCAN 結果（自動找到 {n_clusters} 群）')
plt.show()

### DBSCAN 的威力：月牙形數據

K-Means 遇到非球形的群就徹底失敗，DBSCAN 卻能輕鬆處理！

In [ ]:
from sklearn.datasets import make_moons, make_circles

# 月牙形數據
x_moon, y_moon = make_moons(n_samples=300, noise=0.1, random_state=42)
# 同心圓數據
x_circle, y_circle = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

datasets = [(x_moon, '月牙形'), (x_circle, '同心圓')]

for row, (data, title) in enumerate(datasets):
    # K-Means
    km = KMeans(n_clusters=2, random_state=42, n_init=10)
    km.fit(data)
    axes[row][0].scatter(data[:, 0], data[:, 1],
                         c=km.labels_, cmap='coolwarm', s=30, alpha=0.7)
    axes[row][0].set_title(f'{title} — K-Means（常常失敗）')

    # DBSCAN
    db = DBSCAN(eps=0.2, min_samples=5)
    db.fit(data)
    axes[row][1].scatter(data[:, 0], data[:, 1],
                         c=db.labels_, cmap='coolwarm', s=30, alpha=0.7)
    axes[row][1].set_title(f'{title} — DBSCAN（輕鬆搞定）')

plt.suptitle('K-Means vs DBSCAN：非球形數據的挑戰', fontsize=13)
plt.tight_layout()
plt.show()

## 💡 AI 協作挑戰

**挑戰 1：換個核心函數**

SVM 有不同的核心函數（kernel），試試 `kernel='rbf'`（非線性）和 `kernel='poly'`，看看準確率有沒有改變？

問 AI：不同的 SVM kernel 有什麼差別？什麼時候用哪個？

In [ ]:
kernels = ['linear', 'rbf', 'poly']

for kernel in kernels:
    clf_k = SVC(kernel=kernel)
    clf_k.fit(x_train2, y_train2)
    acc_k = accuracy_score(y_test2, clf_k.predict(x_test2))
    print(f'kernel={kernel:8s}: 測試準確率 = {acc_k:.1%}')

**挑戰 2：用聚類做客群分析**

想像你有一批顧客的消費數據，想把他們分群做精準行銷！

In [ ]:
np.random.seed(87)

# 模擬顧客數據：消費頻率 vs 平均消費金額
n = 200
customers = np.concatenate([
    np.random.multivariate_normal([2, 50], [[0.5, 5], [5, 100]], n//4),    # 低頻低額
    np.random.multivariate_normal([8, 30], [[1, -5], [-5, 50]], n//4),     # 高頻低額
    np.random.multivariate_normal([3, 200], [[0.5, 10], [10, 500]], n//4), # 低頻高額
    np.random.multivariate_normal([9, 180], [[1, 10], [10, 400]], n//4),   # 高頻高額
])

# 用 K-Means 分 4 群
km_customers = KMeans(n_clusters=4, random_state=42, n_init=10)
km_customers.fit(customers)

group_labels = ['低頻低額', '高頻低額', '低頻高額', '高頻高額']

plt.figure(figsize=(8, 6))
scatter = plt.scatter(customers[:, 0], customers[:, 1],
                      c=km_customers.labels_, cmap='viridis', s=60, alpha=0.7)
plt.scatter(km_customers.cluster_centers_[:, 0],
            km_customers.cluster_centers_[:, 1],
            c='red', marker='★', s=400, zorder=5)

plt.xlabel('消費頻率（次/月）')
plt.ylabel('平均消費金額（元）')
plt.title('顧客分群分析（K-Means）')
plt.colorbar(scatter, label='群別')
plt.show()

# 🧠 核心觀念回顧

```
機器學習的兩條路

監督式學習                    非監督式學習
（有標籤）                    （無標籤）
   │                             │
分類問題  回歸問題          聚類  降維  生成
  SVM    線性迴歸        K-Means  PCA  GAN
  決策樹  隨機森林        DBSCAN
  神經網路
```

| 算法 | 類型 | 何時用 |
|------|------|--------|
| SVM | 監督式分類 | 有標籤數據，類別邊界清晰 |
| K-Means | 非監督聚類 | 球形群，知道大概幾群 |
| DBSCAN | 非監督聚類 | 不規則形狀，不知幾群，有離群值 |

# 🎯 本週創作任務

選一個你感興趣的問題，用分類或聚類來解決：

**監督式學習挑戰：**
- 🍷 **葡萄酒分類**：`sklearn.datasets.load_wine()` — 根據化學成分分類葡萄酒
- 💳 **信用卡詐欺偵測**：找公開數據集，訓練一個詐欺偵測分類器

**非監督式學習挑戰：**
- 🗺️ **地圖聚類**：把台灣各行政區按某些特徵聚類
- 🎵 **音樂分類**：把 Spotify 的歌曲按「特徵」（energy, danceability...）分群

**超級挑戰（問 AI）：**
- 試試 `RandomForestClassifier` 或 `GradientBoostingClassifier`
- 比較不同分類器的準確率

回答：

1️⃣ 你選了監督式還是非監督式？為什麼？
2️⃣ 模型效果如何？（準確率 / 視覺化結果）
3️⃣ 發現了什麼有趣的規律？
4️⃣ AI 如何幫助你完成？